# Elaborative Rehearsal (B) — Training (**stage 2**)

Continues from `07`'s checkpoint on `06b`'s rolling data.

## The result that decides whether stage 2 worked

Not ROUGE. §6 drives both checkpoints through `rehearse_elaborative`'s real loop and compares retention by age (C-DIC Fig. 2a).

**Result (flan-t5-base, ratio=0.0, 24 docs, GPU — `scripts/retention_check_gpu.py`)**: early 0.395, late 0.308, drop +0.088. Above the 0.05 collapse threshold, worse than t5-small ratio=0's +0.050, but higher absolute retention than any t5-small config at nearly every age (teacher: 0.383/0.417). Configs: t5-small ratio=0.3 (+0.111), t5-small ratio=0.0 (+0.050), flan-t5-base ratio=0.0 (+0.088).

**Checkpoint-loading bug (2026-08-09/10)**: `AutoModelForSeq2SeqLM.from_pretrained()` re-ties `lm_head.weight` to `shared.weight` when `config.tie_word_embeddings=True`, silently discarding this checkpoint's real (different) `lm_head.weight` — produces fluent-looking garbage. Files themselves are fine. Fix: load normally, then replace just `lm_head.weight` from the checkpoint's safetensors directly (see cell after §1, and `stage1_model` in §6). `tie_word_embeddings=False` on load is too blunt — also un-ties `encoder`/`decoder.embed_tokens`, which should stay tied.


In [16]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.pipeline.curation import probe_accuracy_by_position, retention_probes
from src.pipeline.embeddings import embed_texts, load_config as load_embed_config
from src.pipeline.rehearsal import novel_ngram_ratio, rehearse_elaborative
from src.pipeline.teacher import load_curation_config
from src.pipeline.types import Chunk

CFG = load_curation_config()
EMBED_CFG = load_embed_config()

# Ablation toggle (2026-08-07): self_conditioned_ratio=0.0 (pure teacher
# forcing), testing whether mixing in the stage-1 model's own drifted
# rollout as conditioning (the main run's self_conditioned_ratio=0.3) is
# itself contributing to 07b's collapse signature (+0.111 vs teacher's
# -0.034), rather than only guarding against exposure bias as intended.
# False = exact current behavior, writes/reads the main run's paths
# unchanged. True = separate data dir (built in 06b \u00a79) and separate
# checkpoint dir, so the main run's already-trained model is never touched.
ABLATION_RATIO0 = True

# Base-model toggle (2026-08-08): start stage 2 from the flan-t5-base stage-1
# checkpoint (07, MODEL_NAME swapped from t5-small) instead of t5-small.
# Combined with ABLATION_RATIO0=True since that setting is already confirmed
# better (halved the collapse drop) -- no reason to re-run the flan-t5-base
# ablation against the worse ratio=0.3 setting.
BASE_MODEL_FLANT5 = True

STAGE1_DIR = (
    Path("experiments/rehearsal_elaborative_flant5base")
    if BASE_MODEL_FLANT5
    else Path("experiments/rehearsal_elaborative_small")
)
CACHE_DIR = Path("data/processed/rehearsal_elaborative_stage2")  # curation_cache.jsonl always lives here -- curation itself is never re-run for the ablation, only the pairs are rebuilt at a different ratio
if ABLATION_RATIO0:
    DATA_DIR = Path("data/processed/rehearsal_elaborative_stage2_ratio0")
else:
    DATA_DIR = Path("data/processed/rehearsal_elaborative_stage2")

_output_suffix = ("_flant5base" if BASE_MODEL_FLANT5 else "") + ("_ratio0" if ABLATION_RATIO0 else "")
OUTPUT_DIR = Path(f"experiments/rehearsal_elaborative_stage2{_output_suffix}")

DATA_READY = DATA_DIR.exists() and (DATA_DIR / "train").exists()
STAGE1_READY = STAGE1_DIR.exists()
print(f"ABLATION_RATIO0  : {ABLATION_RATIO0}")
print(f"BASE_MODEL_FLANT5: {BASE_MODEL_FLANT5}")
print(f"stage-2 data     : {'ready' if DATA_READY else f'MISSING — run 06b (\u00a79 if ABLATION_RATIO0)'} ({DATA_DIR})")
print(f"stage-1 ckpt     : {'ready' if STAGE1_READY else 'MISSING — run 07 first'} ({STAGE1_DIR})")
print(f"output dir       : {OUTPUT_DIR}")

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅
ABLATION_RATIO0  : True
BASE_MODEL_FLANT5: True
stage-2 data     : ready (data/processed/rehearsal_elaborative_stage2_ratio0)
stage-1 ckpt     : ready (experiments/rehearsal_elaborative_flant5base)
output dir       : experiments/rehearsal_elaborative_stage2_flant5base_ratio0


## 1. Load

Initialised from the stage-1 checkpoint, not `t5-small` — preserves the abstraction it already learned.


In [17]:
if not (DATA_READY and STAGE1_READY):
    raise SystemExit("run 06b and 07 first")

dataset = load_from_disk(str(DATA_DIR))
tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)

print(dataset)
frame = pd.read_csv(DATA_DIR / "train_pairs_raw.csv")
print(f"\nself-conditioned share: {(frame['conditioning'] == 'self').mean():.1%}")
print(frame["genre"].value_counts().to_string())

DatasetDict({
    train: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3742
    })
    val: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 270
    })
    test: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 273
    })
})

self-conditioned share: 0.0%
genre
wiki           978
narrativeqa    946
caselaw        914
news           904


In [ ]:
# Stage 2 was already trained on a rented GPU (scripts/train_stage2_gpu.py,
# vast.ai) -- skip §3's trainer.train() entirely and load the already-trained
# checkpoint straight from OUTPUT_DIR instead of STAGE1_DIR. §6 below only
# needs `model`/`tokenizer` in scope; it doesn't care how they got here.
#
# BUG (2026-08-09/10): AutoModelForSeq2SeqLM.from_pretrained() re-ties
# lm_head.weight to shared.weight on load because config.tie_word_embeddings=True,
# discarding the checkpoint's actually-trained (and different -- flan-t5-base
# is not really tied despite the config flag) lm_head.weight. Silent -- no
# error, no warning -- and produces fluent-looking garbage (repeats rare
# unrelated subwords) since the output layer becomes an embedding matrix that
# was never optimized as a vocab projection. The safetensors file itself was
# never corrupted (lm_head.weight is present and correct on disk, distinct
# from shared.weight).
#
# Fix: load normally (so encoder/decoder embed_tokens stay correctly tied to
# shared.weight -- that tie IS structural to T5 and must survive), then
# surgically replace just lm_head.weight with the value read straight from
# the checkpoint's safetensors file. Passing config.tie_word_embeddings=False
# to from_pretrained() is tempting but too blunt: on some transformers
# versions (confirmed on the vast.ai remote's) it also un-ties
# encoder/decoder embed_tokens, which are never saved as separate keys in a
# T5 checkpoint, so they come back MISSING and get randomly reinitialized --
# same fluent-looking-garbage failure mode, just from a different cause.
from safetensors.torch import load_file

model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_state = load_file(str(OUTPUT_DIR / "model.safetensors"))
if "lm_head.weight" in _state:
    model.lm_head.weight = torch.nn.Parameter(_state["lm_head.weight"].to(dtype=model.lm_head.weight.dtype))
    model.config.tie_word_embeddings = False
assert model.lm_head.weight is not model.shared.weight, "lm_head got re-tied -- fix didn't take"
print(f"loaded trained stage-2 checkpoint from {OUTPUT_DIR}, parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Metrics

ROUGE vs teacher summary + novel-3-gram ratio (keeps B honest — it must rewrite, not lift verbatim).


In [19]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(predictions=decoded_predictions, references=decoded_labels)
    scores["novel_3gram"] = float(
        np.mean([novel_ngram_ratio(p, r, n=3) for p, r in zip(decoded_predictions, decoded_labels)])
    )
    scores["gen_len"] = float(np.mean([len(p.split()) for p in decoded_predictions]))
    return {k: round(float(v), 4) for k, v in scores.items()}

## 3. Train

Small learning rate — continued training, preserving stage-1 abstraction.


In [5]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    predict_with_generate=True,
    generation_max_length=256,
    logging_steps=50,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

trainer.train()

  0%|          | 0/2808 [00:00<?, ?it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
  2%|▏         | 50/2808 [01:04<1:00:54,  1.32s/it]

{'loss': 2.0759, 'grad_norm': 2.1384637355804443, 'learning_rate': 9.821937321937323e-05, 'epoch': 0.05}


  4%|▎         | 100/2808 [02:06<57:44,  1.28s/it] 

{'loss': 1.3175, 'grad_norm': 2.2229394912719727, 'learning_rate': 9.643874643874644e-05, 'epoch': 0.11}


  5%|▌         | 150/2808 [03:09<54:22,  1.23s/it]

...

100%|██████████| 2808/2808 [1:17:25<00:00,  1.33s/it]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                     
100%|██████████| 2808/2808 [1:24:41<00:00,  1.33s/it]

{'eval_loss': 0.7847525477409363, 'eval_rouge1': 0.5975, 'eval_rouge2': 0.5051, 'eval_rougeL': 0.5449, 'eval_rougeLsum': 0.5426, 'eval_novel_3gram': 0.456, 'eval_gen_len': 141.437, 'eval_runtime': 433.1825, 'eval_samples_per_second': 0.623, 'eval_steps_per_second': 0.157, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
100%|██████████| 2808/2808 [1:24:44<00:00,  1.81s/it]

{'train_runtime': 5084.0031, 'train_samples_per_second': 2.208, 'train_steps_per_second': 0.552, 'train_loss': 0.9619423480455013, 'epoch': 3.0}


TrainOutput(global_step=2808, training_loss=0.9619423480455013, metrics={'train_runtime': 5084.0031, 'train_samples_per_second': 2.208, 'train_steps_per_second': 0.552, 'total_flos': 1514917791399936.0, 'train_loss': 0.9619423480455013, 'epoch': 3.0})

In [6]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("saved to", OUTPUT_DIR)

saved to experiments/rehearsal_elaborative_stage2_ratio0


## 4. Held-out test set

Touched once; the checkpoint was already selected on validation.


In [7]:
test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
pd.Series(test_metrics).to_frame("value")

/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 69/69 [07:41<00:00,  6.68s/it]


,value
test_loss,0.816729
test_rouge1,0.648200
test_rouge2,0.569100
test_rougeL,0.604200
test_rougeLsum,0.602600
test_novel_3gram,0.375200
test_gen_len,142.300400
test_runtime,468.851100
test_samples_per_second,0.582000
test_steps_per_second,0.147000


## 5. Where does the model sit on the insert/revise branch?

Measured as growth: output length minus conditioning-summary length. Append-only grows every step; revising stays flat.


In [24]:
from tqdm.auto import tqdm

test_frame = pd.read_csv(DATA_DIR / "test_pairs_raw.csv")
sample = test_frame[test_frame["chunk_position"] > 0].head(60)

device = next(model.parameters()).device
model.eval()

rows = []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="§5 growth check"):
    inputs = tokenizer(row["input_text"], return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    rows.append({
        "chunk_position": row["chunk_position"],
        "prev_words": len(str(row["input_text"]).split()),
        "pred_words": len(prediction.split()),
        "target_words": len(str(row["target_text"]).split()),
    })

growth = pd.DataFrame(rows)
print(growth.groupby("chunk_position")[["pred_words", "target_words"]].mean().round(1).to_string())
print("\nIf pred_words climbs with chunk_position while target_words does not,")
print("the model is appending rather than revising — Eq. 6's revise branch did not transfer.")

§5 growth check: 100%|██████████| 60/60 [58:48<00:00, 58.80s/it]   

                pred_words  target_words
chunk_position                          
1                    127.8         138.0
2                    158.0         106.2
3                    182.2         176.5
4                    136.8          85.3
5                    145.7          98.5
6                    185.4         111.8
7                    163.6         102.8
8                    155.4         118.4
9                    125.4          87.0
10                   170.2         131.8
11                   154.0         138.2

If pred_words climbs with chunk_position while target_words does not,
the model is appending rather than revising — Eq. 6's revise branch did not transfer.


## 6. The check that actually matters — collapse in the real loop

Both checkpoints driven through `rehearse_elaborative`'s real loop; retention measured by age (C-DIC Fig 2a).


In [ ]:
import json

from tqdm.auto import tqdm

records = {}
for line in (CACHE_DIR / "curation_cache.jsonl").read_text(encoding="utf-8").splitlines():
    if line.strip():
        record = json.loads(line)
        records[record["key"]] = record

test_keys = set(pd.read_csv(DATA_DIR / "test_pairs_raw.csv")["key"])
test_records = [records[k] for k in test_keys if k in records]
print(f"{len(test_records)} held-out documents")

# Same lm_head fix as the OUTPUT_DIR load above -- STAGE1_DIR hits the
# identical bug when reloaded fresh here.
stage1_model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)
stage1_tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)
_stage1_state = load_file(str(STAGE1_DIR / "model.safetensors"))
if "lm_head.weight" in _stage1_state:
    stage1_model.lm_head.weight = torch.nn.Parameter(
        _stage1_state["lm_head.weight"].to(dtype=stage1_model.lm_head.weight.dtype)
    )
    stage1_model.config.tie_word_embeddings = False
assert stage1_model.lm_head.weight is not stage1_model.shared.weight, "lm_head got re-tied -- fix didn't take"


def memory_states_for(m, tok, chunk_texts: list[str]) -> list[str]:
    """Drives the real ThreadMemory loop and returns, per position, every
    slot's text concatenated — what a downstream reader would see at that
    point. Not `rolling_self_summaries` (retrieval-free): that function
    approximates a single-slot rollout, which is no longer the shape stage 2
    was trained on (06b, 2026-08-04) — the retrieval half has to run too, or
    this check would validate a loop the model was never actually asked to
    perform."""
    chunks = [Chunk(text=t, index=i) for i, t in enumerate(chunk_texts)]
    _, recs = rehearse_elaborative(
        chunks, m, tok, EMBED_CFG, embed_texts,
        max_new_tokens=256, record_memory_state=True,
    )
    return [r.memory_state for r in recs]


curves = {}
for name, (m, tok) in {
    "stage 1 (one-shot, driven incrementally)": (stage1_model, stage1_tokenizer),
    "stage 2 (rolling, threaded)": (model, tokenizer),
    "teacher (upper bound)": (None, None),
}.items():
    probes = []
    for record in tqdm(test_records, desc=name):
        answers = {int(k): v for k, v in record["answers_by_chunk"].items()}
        # The teacher's own upper bound is its memory_states, not gists[i] —
        # same reasoning as 06b §5: a fact can live in a slot the current
        # chunk never touches.
        states = record["memory_states"] if m is None else memory_states_for(m, tok, record["chunk_texts"])
        probes += retention_probes(states, answers, CFG["probe_f1_threshold"])
    report = probe_accuracy_by_position(probes, collapse_after=3)
    curves[name] = report
    print(f"{name:45} early {report['early_accuracy']:.3f}  late {report['late_accuracy']:.3f}  drop {report['drop']:+.3f}")

comparison = pd.DataFrame({name: report["by_position"] for name, report in curves.items()})
comparison.index.name = "age (compressions since the fact was read)"
display(comparison.round(3))

Path("results").mkdir(exist_ok=True)
_retention_suffix = ("_flant5base" if BASE_MODEL_FLANT5 else "") + ("_ratio0" if ABLATION_RATIO0 else "")
retention_out_path = f"results/elaborative_stage2_retention_by_age{_retention_suffix}.csv"
comparison.to_csv(retention_out_path)
print(f"saved to {retention_out_path}")

## Summary

| stage-2 checkpoint | early | late | drop |
|---|---|---|---|
| t5-small, ratio=0.3 (original) | 0.355 | 0.244 | +0.111 |
| t5-small, ratio=0.0 (ablation) | 0.312 | 0.263 | +0.050 |
| **flan-t5-base, ratio=0.0 (current)** | **0.395** | **0.308** | **+0.088** |
| *(ref)* stage 1 | 0.065 | 0.087 | -0.022 |
| *(ref)* teacher | 0.383 | 0.417 | -0.034 |

Removing self-conditioning (ratio=0.0) roughly halves drop. Scaling the base model raises absolute retention but doesn't stack with the ratio fix on drop (+0.088 > +0.050). Bigger model = higher ceiling, same-shaped curve.

§5 (growth check): too noisy to read standalone.

**Does not**: validate `ThreadMemory` capacity/overflow policy (§6 uses `memory_capacity=None`).

**Does not (yet)**: confirm flan-t5-base's retention gain helps downstream QA — `10` still points at t5-small ratio=0.0.

**Caveats**: teacher is an upper bound on this pipeline, not the task. Self-conditioned mix is DAgger-like, not ra-TBPTT. All numbers above use the fixed lm_head loading path.
